# remote

> Hosted models through [fastllm](https://github.com/AnswerDotAI/fastllm) — same `Chat` API with API keys in the environment (`ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, `GEMINI_API_KEY`, …).

Portable `hist` lets you start local and hand off to Claude/GPT/Gemini without reformatting.


In [ ]:
#| default_exp remote

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json, os, asyncio
from base64 import b64encode
from fastllm.acomplete import acomplete
from fastllm.types import Completion, Usage
from aidialog.msg_parts import Msg, Part, PartType, data_url
from fastcore.funccall import mk_ns
from fastcore.all import Path, store_attr, patch, L, ifnone, first, listify
from rishi import core
from rishi.core import *

/Users/71293/code/personal/orgs/rishi/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#| export
_all_ = ['UsageStats', 'ChatCallback', 'run_cbs', 'resp_text', 'thought', 'Resp', 'StreamFormatter',
         'display_stream', 'mk_tr_details', 'truncated', 'hitl_policy', 'extract_fence', 'mk_toolspec',
         'ToolCall', 'mk_tool_res_msg']

In [ ]:
from fastcore.test import test_eq, test_fail, test_close

## Messages

fastllm's canonical message is a `Msg(role, content=[Part, ...])`, where a `Part` is typed: `text`,
`thinking`, `tool_use`, `tool_result`, `input_image`/`input_audio` (whose payload is a data URL in
`part.text`). rishi's canonical history is OpenAI-shaped dicts. Neither is richer than the other in
practice, so this is a straight two-way mapping - and it is where media fidelity across a backend hop
actually gets decided, so images and audio are carried through as data URLs rather than collapsed to
a placeholder.

Note thinking round-trips properly here: rishi keeps it in `channels.thought`, fastllm keeps it as a
`thinking` Part, and each converts to the other.

In [ ]:
#| export
def _img_url(p):
    "Data URL for an OpenAI-style image part (`image_url` may be a dict or a bare string)."
    u = p.get('image_url')
    return u.get('url') if isinstance(u, dict) else u

def _aud_url(p):
    "Data URL for an OpenAI-style `input_audio` part."
    a = p.get('input_audio') or {}
    return f"data:audio/{a.get('format', 'wav')};base64,{a.get('data', '')}"

def _parts(content):
    "Canonical rishi content (str or list of parts) -> fastllm `Part`s."
    if content is None: return []
    if isinstance(content, str): return [Part(type=PartType.text, text=content)]
    out = []
    for p in content:
        if not isinstance(p, dict): continue
        t = p.get('type')
        if   t == 'text':        out.append(Part(type=PartType.text, text=p.get('text', '')))
        elif t == 'image_url':   out.append(Part(type=PartType.input_image, text=_img_url(p)))
        elif t == 'input_audio': out.append(Part(type=PartType.input_audio, text=_aud_url(p)))
    return out

def to_msg(m):
    "One canonical rishi history dict -> a fastllm `Msg`."
    role = m.get('role', 'user')
    if role == 'tool':
        return Msg(role='tool', content=[Part(type=PartType.tool_result, text=str(m.get('content', '')),
                                              data={'id': m.get('tool_call_id'), 'name': m.get('name', '')})])
    parts = []
    if role == 'assistant' and (th := (m.get('channels') or {}).get('thought')):
        parts.append(Part(type=PartType.thinking, text=th))
    parts += _parts(m.get('content'))
    for tc in (m.get('tool_calls') or []):
        fn = tc.get('function') or {}
        parts.append(Part(type=PartType.tool_use, data={'id': tc.get('id'), 'name': fn.get('name', ''),
                                                        'arguments': fn.get('arguments') or {},
                                                        'server': bool(tc.get('server'))}))
    return Msg(role=role, content=parts)

def to_hist(m):
    "A fastllm `Msg` -> canonical rishi history dicts (a tool `Msg` can hold several results)."
    if m.role == 'tool':
        return [{'role': 'tool', 'tool_call_id': (p.data or {}).get('id'), 'name': (p.data or {}).get('name', ''),
                 'content': str(p.text)} for p in m.content if p.type == PartType.tool_result]
    text = ''.join(p.text or '' for p in m.content if p.type == PartType.text)
    th   = ''.join(p.text or '' for p in m.content if p.type == PartType.thinking)
    media = [p for p in m.content if p.type in (PartType.input_image, PartType.input_audio)]
    out = {'role': m.role, 'content': text}
    if media:
        parts = ([{'type': 'text', 'text': text}] if text else []) + [_media_part(p) for p in media]
        out['content'] = parts
    if th: out['channels'] = {'thought': th}
    tcs = [ToolCall(name=(p.data or {}).get('name', ''), arguments=(p.data or {}).get('arguments') or {},
                    id=(p.data or {}).get('id'), server=bool((p.data or {}).get('server')))
           for p in m.content if p.type == PartType.tool_use]
    if tcs: out['tool_calls'] = tcs
    return [out]

def _media_part(p):
    "A fastllm media `Part` -> an OpenAI-style content part."
    if p.type == PartType.input_image: return {'type': 'image_url', 'image_url': {'url': p.text}}
    mime, b64 = data_url(p.text) or ('audio/wav', '')
    return {'type': 'input_audio', 'input_audio': {'data': b64, 'format': mime.split('/')[-1]}}

In [ ]:
# a round trip through fastllm's Msg keeps text, thinking, tool calls and tool results intact
h = [{'role': 'user', 'content': 'what is 2+2?'},
     {'role': 'assistant', 'content': 'let me add', 'channels': {'thought': 'hmm'},
      'tool_calls': [ToolCall('add', {'a': 2, 'b': 2}, id='c1')]},
     {'role': 'tool', 'tool_call_id': 'c1', 'name': 'add', 'content': '4'},
     {'role': 'assistant', 'content': 'it is 4'}]
msgs = [to_msg(m) for m in h]
test_eq([m.role for m in msgs], ['user', 'assistant', 'tool', 'assistant'])
back = [x for m in msgs for x in to_hist(m)]
test_eq(back[0], {'role': 'user', 'content': 'what is 2+2?'})
test_eq(back[1]['channels'], {'thought': 'hmm'})
test_eq(back[1]['tool_calls'][0].name, 'add')
test_eq(back[1]['tool_calls'][0].arguments, {'a': 2, 'b': 2})
test_eq(back[2], {'role': 'tool', 'tool_call_id': 'c1', 'name': 'add', 'content': '4'})
test_eq(back[3], {'role': 'assistant', 'content': 'it is 4'})

# media survives the hop as a data URL, rather than being collapsed to a placeholder
png = b'\x89PNG\r\n\x1a\n' + b'0' * 8
um = mk_oai_msg([png, 'what is this?'])
m = to_msg(um)
test_eq([p.type for p in m.content], ['input_image', 'text'])
assert m.content[0].text.startswith('data:image/png;base64,')
rt = to_hist(m)[0]
test_eq(rt['content'][1]['image_url']['url'], m.content[0].text)

# audio too
wav = mk_oai_msg([b'RIFF0000WAVE', 'transcribe'])
am = to_msg(wav)
test_eq([p.type for p in am.content], ['input_audio', 'text'])
test_eq(to_hist(am)[0]['content'][1]['input_audio']['format'], 'wav')

# a server-side tool call keeps its flag across the hop
sm = to_msg({'role': 'assistant', 'content': '', 'tool_calls': [ToolCall('web_search', {}, id='s1', server=True)]})
assert to_hist(sm)[0]['tool_calls'][0].server

## Responses and usage

`norm_completion` maps fastllm completions to rishi `Resp`. `UsageStats.cached_tokens` is populated when the provider reports prompt caching.


In [ ]:
#| export
def norm_usage(u, model=None):
    "fastllm `Usage` -> rishi `UsageStats`."
    if u is None: return {}
    return {'prompt_tokens': u.prompt_tokens, 'completion_tokens': u.completion_tokens,
            'total_tokens': u.total_tokens or (u.prompt_tokens + u.completion_tokens),
            'cached_tokens': u.cached_tokens, 'model': model}

def norm_completion(comp):
    "fastllm `Completion` -> rishi `Resp`, with `<tool_call>` tags read out of the text as `core.norm_resp` does."
    res = to_hist(comp.message)[0]
    res.setdefault('role', 'assistant')
    tag_tcs = []
    if isinstance(res.get('content'), str): res['content'], tag_tcs = parse_tool_tags(res['content'])
    tcs = [ToolCall(name=tc.name, arguments=tc.arguments, id=tc.id, server=tc.server) for tc in (comp.tool_calls or [])]
    tcs += [ToolCall(name=tc['function']['name'], arguments=tc['function']['arguments'], id=tc['id']) for tc in tag_tcs]
    if tcs: res['tool_calls'] = tcs
    if comp.finish_reason == 'length': res['truncated'] = True
    res['usage'] = norm_usage(comp.usage, comp.model)
    return Resp(res)

## RemoteChat

Async wire via `run_coro` / `sync_iter`. Hosted-only passthrough: `tool_choice`, `reasoning_effort`, and `tool_mode='tags'|'native'`. Provider-run tools return `server=True` and are recorded, not executed locally.


In [ ]:
#| export
#: What `think=False` asks a hosted model for - a name, not a literal, so a deployment whose
#: provider spells the lowest reasoning effort differently can say so.
NO_THINK_EFFORT = 'minimal'
class RemoteChat(ToolLoopMixin, Chat):
    "Chat against a hosted model through fastllm - the same `rishi.core.Chat` API as the local backends."
    _runtime = 'remote'
    _dflt_cbs = [UsageCallback, ToolReminderCallback, SlidingWindowCallback]
    mk_content, mk_msg, mk_msgs = staticmethod(mk_oai_content), staticmethod(mk_oai_msg), staticmethod(mk_oai_msgs)

    @staticmethod
    def fmt2hist(msgs):
        "fastllm `Msg`s (or canonical dicts) -> canonical rishi history dicts."
        out = []
        for m in listify(msgs): out += to_hist(m) if isinstance(m, Msg) else [mk_oai_msg(m)]
        return out
    @staticmethod
    def hist2fmt(msgs):
        "Canonical rishi history dicts -> fastllm `Msg`s (media carried through, not stripped)."
        return [to_msg(m) for m in listify(msgs) if m.get('role') != 'system']

    def __init__(self, model=None, *, runtime=None, model_path=None, api_key=None, base_url=None,
                 vendor_name=None, api_name=None, sp='', messages=None, tools=None, ctx_limit=None,
                 approve=None, tool_max_len=None, max_steps=10, parallel_tools=False,
                 max_parallel_tools=None, final_prompt=dflt_final_prompt_, tool_choice=None, reasoning_effort=None,
                 temp=None, max_output_tokens=4096, retries=2, comp_kw=None, cbs=None, default_cbs=True,
                 tool_mode='native'):   # 'native' sends schemas on the wire; 'tags' puts them in the system prompt
        model = core.split_runtime(model)[1]
        self.tool_mode = tool_mode
        self.model_id = model or 'gpt-5.1'
        self._set_tools(tools)
        store_attr('api_key,base_url,vendor_name,api_name,tool_choice,reasoning_effort,temp,'
                   'max_output_tokens,retries', self)
        self.comp_kw, self._ctx_tokens = comp_kw or {}, 0
        self.ctx_limit = ctx_limit
        self._setup(model=model, sp=sp, messages=messages, tools=tools, approve=approve,
                    tool_max_len=tool_max_len, max_steps=max_steps, parallel_tools=parallel_tools,
                    max_parallel_tools=max_parallel_tools, final_prompt=final_prompt, cbs=cbs, default_cbs=default_cbs)

    @property
    def token_count(self):
        "Tokens the last turn reported (prompt + completion); hosted APIs have no live context read-out."
        return self._ctx_tokens

    def _kw(self, stream=False, max_output_tokens=None):
        "Keyword arguments for `acomplete`."
        kw = dict(stream=stream, api_key=self.api_key, base_url=self.base_url, retries=self.retries,
                  vendor_name=self.vendor_name, api_name=self.api_name,
                  max_tokens=ifnone(max_output_tokens, self.max_output_tokens))
        tags = self.tool_mode == 'tags'
        sp = tag_tools_sp(self.toolspecs, self.sp) if tags else self.sp
        if sp: kw['system'] = sp
        # In tag mode the schemas have already gone out in the system prompt, and putting them
        # on the wire as well is the thing the transport cannot do.
        if self.toolspecs and not tags: kw['tools'] = self.toolspecs
        if self.tool_choice is not None and not tags: kw['tool_choice'] = self.tool_choice
        if self.reasoning_effort is not None: kw['reasoning_effort'] = self.reasoning_effort
        if self.temp is not None: kw['temperature'] = self.temp
        return {**kw, **self.comp_kw}

    async def _acomplete(self, msgs, stream=False, max_output_tokens=None):
        "One `acomplete` call for `msgs` (canonical rishi dicts)."
        return await acomplete(self.hist2fmt(msgs), self.model_id, **self._kw(stream, max_output_tokens))

    def _model_step(self, max_output_tokens=None):
        "One completion, normalized to a `Resp` - the wire call `ToolLoopMixin` drives."
        return norm_completion(run_coro(self._acomplete(self.hist, False, max_output_tokens)))

    def _stream_step(self, max_output_tokens=None):
        "Stream one completion; yields chunk dicts and leaves the merged `Resp` on `self._step_res`."
        async def _agen():
            agen = await self._acomplete(self.hist, True, max_output_tokens)
            async for o in agen: yield o
        comp, split = None, StreamSplit() if self.tool_mode == 'tags' else None
        for o in sync_iter(_agen):
            if isinstance(o, Completion): comp = o
            elif isinstance(o, Part):
                if o.type == PartType.tool_use and o.data:
                    yield {'content': [{'type': 'tool_call', 'name': o.data.get('name', ''),
                                        'arguments': o.data.get('arguments') or {}}]}
            elif isinstance(o, dict):
                # In tag mode the calls arrive as text, so the text is split on the way past --
                # otherwise a `<tool_call>` block renders in the transcript as prose before the
                # final `Resp` quietly turns it into a call.
                if (t := o.get('text')): yield from (split.feed(t) if split else
                                                     [{'content': [{'type': 'text', 'text': t}]}])
                if (th := o.get('thinking')): yield {'channels': {'thought': th}}
        if split is not None: yield from split.finish()
        if comp is None: raise RuntimeError('stream ended without a final Completion')
        self._step_res = norm_completion(comp)

    def _oneshot(self, prompt, sp='', think=None, max_tokens=None):
        "Stateless one-shot completion text (no history, no tools); `think=False` asks for `NO_THINK_EFFORT`."
        msgs = [to_msg({'role': 'user', 'content': prompt})]
        kw = {**self._kw(max_output_tokens=max_tokens), 'system': sp or None, 'tools': None, 'tool_choice': None}
        send = lambda k: resp_text(norm_completion(run_coro(acomplete(msgs, self.model_id, **k))))
        if think is not False: return send(kw)
        try: return send({**kw, 'reasoning_effort': NO_THINK_EFFORT})
        except Exception: return send(kw)   # a provider that spells the effort differently

    def _structured_call(self, prompt, schema, sp):
        "Force the tool call for `schema` and return its arguments; falls back to parsing JSON from prose."
        spec = mk_toolspec(schema)
        name = spec['function']['name']
        kw = {**self._kw(), 'system': sp or None, 'tools': [spec], 'tool_choice': name}
        comp = run_coro(acomplete([to_msg({'role': 'user', 'content': prompt})], self.model_id, **kw))
        r = norm_completion(comp)
        if tcs := r.get('tool_calls'): return tcs[0].arguments
        txt = resp_text(r)
        try: return json.loads(extract_fence(txt, 'json'))
        except (json.JSONDecodeError, TypeError):
            raise ValueError(f"model neither called the tool nor returned JSON; reply: {txt[:200]!r}")

    def close(self):
        "Nothing to release - the HTTP client is fastllm's, and cached across chats."
        pass

In [ ]:
def _add(a: int, b: int) -> int:
    'Add a and b.'
    return a + b

# native puts the schemas on the wire; tags puts them in the system prompt and leaves the field empty
kw = RemoteChat('gpt-5.1', tools=[_add], sp='Be terse.')._kw()
test_eq([t['function']['name'] for t in kw['tools']], ['_add'])
test_eq(kw['system'], 'Be terse.')

kw = RemoteChat('gpt-5.1', tools=[_add], sp='Be terse.', tool_mode='tags', tool_choice='required')._kw()
assert 'tools' not in kw and 'tool_choice' not in kw
assert kw['system'].startswith('Be terse.') and '"name": "_add"' in kw['system']

# ...and a tag call in the reply text comes back as a real tool call, with the prose left behind
def _comp(text):
    return Completion(model='m', usage=Usage(prompt_tokens=1, completion_tokens=2),
                      message=Msg(role='assistant', content=[Part(type=PartType.text, text=text)]))
r = norm_completion(_comp('on it\n<tool_call>{"name": "_add", "arguments": {"a": 1, "b": 2}}</tool_call>'))
test_eq(resp_text(r), 'on it')
test_eq((r['tool_calls'][0].name, r['tool_calls'][0].arguments), ('_add', {'a': 1, 'b': 2}))
test_eq(norm_completion(_comp('just prose')).get('tool_calls'), None)

## Against a real API

`Chat('gpt-5.1')`, `Chat('anthropic/claude-sonnet-4-5')`, or `runtime='remote'` — routing only; you need a vendor key.


In [ ]:
#| eval: false
from fastllm.acomplete import acomplete               # undo the test patch above

chat = Chat('gpt-4.1', sp='You are concise.')
test_eq(chat.runtime, 'remote')
r = chat('Reply with exactly: pong')
assert 'pong' in resp_text(r).lower()
print(chat.use)

total=22|in=20|out=2|turns=1|model=gpt-4.1


In [ ]:
#| eval: false
display_stream(Chat('gpt-5.1')('Write two sentences about the monsoon.', stream=True))

In [ ]:
#| eval: false
th = Chat('gpt-5.1', reasoning_effort='high', max_output_tokens=2048)
r = th('A bat and ball cost $1.10, the bat is $1 more than the ball. How much is the ball?')
print(thought(r)[:400]); print('---'); print(resp_text(r))

In [ ]:
#| eval: false
img = Path('images.jpeg').read_bytes()
print(resp_text(Chat('gpt-5.1')([img, 'What is in this image? One sentence.'])))

A German Shepherd dog is standing on an outdoor path with its tongue out.


In [ ]:
#| eval: false
# the whole point: start local, finish hosted, with one history
from rishi.llama import LlamaChat, qwen3_17b

local = LlamaChat(qwen3_17b, n_ctx=4096)
local('My name is Karthik and my favourite number is 17. Remember both.')
hist = local.hist
local.close()

remote = Chat('gpt-5.1', messages=hist)
r = remote('What is my name and my favourite number?')
assert 'karthik' in resp_text(r).lower() and '17' in resp_text(r)
resp_text(r)

llama_context: n_ctx_seq (4096) < n_ctx_train (40960) -- the full capacity of the model will not be utilized


'Your name is Karthik, and your favourite number is 17.'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()